### Название скрипта: Получение списка городов РФ со страницы Википедии

#### Описание:
    Скрипт для первичного сбора и обновления данных о городах
    РФ и их сохранения в csv-файл / базу данных для дальнейшего
    использования в работе.

#### Основные функции:
    - Подключение к Википедии
    - Получение страницы со списком городов
    - Парсинг страницы
    - Обработка данных
    - Подключение к библиотеке Geopy
    - Получение геоданных городов
    - Контрольная проверка
    - Сохранение обработанных данных

#### Использование:
    Goroda_Russia_all.csv название файла

#### Требования:
    - Python 3.10+
    - Зависимости: pandas, requests

In [1]:
!pip install geopy

In [2]:
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim
from pprint import pprint
from io import StringIO
from tqdm import tqdm
import pandas as pd
import requests
import warnings
import time

In [3]:
# Используем Wikipedia API для получения данных страницы в JSON
api_url = "https://ru.wikipedia.org/w/api.php"
params = {
    'action': 'parse',
    'page': 'Список_городов_России',
    'format': 'json',
    'prop': 'text',
    'section': 1  # Обычно таблица находится в первом разделе
}

In [4]:
# Делаем запрос к странице с параметрами прописанными выше
response = requests.get(api_url, params=params)

In [5]:
# забираем ответ в JSON
data = response.json()

In [6]:
# Для отладки сложных JSON-структур используем функцию pprint, которая позволяет наглядно отобразить данные
# pprint(data)

In [7]:
# Конвертируем в HTML код
# В этом выражении data — объект, содержащий данные, которые нужно обработать как HTML.
# parse — метод, который преобразует данные в формат HTML. text — свойство, возвращающее текст элементов HTML
html = data['parse']['text']['*']
# pprint(html)

In [8]:
# Получаем список датасетов (таблицы)
# [0] потому что read_html возвращает список фреймов данных
dfs_string_io = pd.read_html(StringIO(html))[0]
dfs_string_io

,№,Герб,Город,Регион,Федеральный округ,Население,Основание или первое упоминание,Статус города[3],Прежние названия
0,1,NaN,Абаза,Хакасия,Сибирский,12 272,1867,1966,"Абаканский Завод, Абаканско-Заводское"
1,2,NaN,Абакан,Хакасия,Сибирский,184 769,1734,1931,Усть-Абаканское (до 1931)
2,3,NaN,Абдулино,Оренбургская область,Приволжский,17 274,1795,1923,NaN
3,4,NaN,Абинск,Краснодарский край,Южный,39 511,1863,1963,Абинское (до 1863); Абинская (до 1962)
4,5,NaN,Агидель,Башкортостан,Приволжский,14 219,1980,1991,NaN
...,...,...,...,...,...,...,...,...,...
1120,1121,NaN,Ярославль,Ярославская область,Центральный,577 279,1010,1071,NaN
1121,1122,NaN,Ярцево,Смоленская область,Центральный,41 452,1610,1926,NaN
1122,1123,NaN,Ясногорск,Тульская область,Центральный,15 269,1578,1958,Лаптево (до 1965)
1123,1124,NaN,Ясный,Оренбургская область,Приволжский,15 471,1961,1979,NaN


In [9]:
# Удаляем не нужные столбцы
df = dfs_string_io.drop(['№', 'Герб', 'Федеральный округ', 'Прежние названия'], axis=1)
df.head()

,Город,Регион,Население,Основание или первое упоминание,Статус города[3]
0,Абаза,Хакасия,12 272,1867,1966
1,Абакан,Хакасия,184 769,1734,1931
2,Абдулино,Оренбургская область,17 274,1795,1923
3,Абинск,Краснодарский край,39 511,1863,1963
4,Агидель,Башкортостан,14 219,1980,1991


In [10]:
# Переименовываем колонки для удобства
# создаем список из текущих колонок
old_cols = df.columns.tolist()
# список новых
new_cols = ['city', 'region', 'population', 'found', 'status']

# создаем словарь старая колонка - новая колонка
cols_dict = dict(zip(old_cols, new_cols))

# переименовываем через rename
df = df.rename(columns=cols_dict)
df.head()

,city,region,population,found,status
0,Абаза,Хакасия,12 272,1867,1966
1,Абакан,Хакасия,184 769,1734,1931
2,Абдулино,Оренбургская область,17 274,1795,1923
3,Абинск,Краснодарский край,39 511,1863,1963
4,Агидель,Башкортостан,14 219,1980,1991


In [11]:
# Есть сомнения как считались города Крыма, т.к. они с примечаниями в таблице. Проверю:
df.query('region == "Крым"')

,city,region,population,found,status
22,Алупкане призн.,Крым,9063,960,1938
23,Алуштане призн.,Крым,31 364,VI век,1902
42,Армянскне призн.,Крым,20 692,1736,1993
76,Бахчисарайне призн.,Крым,28 609,1532,1532
86,Белогорскне призн.,Крым,17 445,XIII век,XIII век
250,Джанкойне призн.,Крым,37 014,1784,1917
274,Евпаторияне призн.,Крым,107 877,497 год до н. э.,1784
397,Керчьне призн.,Крым,154 621,VII—VI век до н. э.,NaN
474,Красноперекопскне призн.,Крым,25 569,1932,1966
825,Сакине призн.,Крым,24 285,1952,1952


ч.т.д.

Нужно с эти поработать.

In [12]:
# удалю из названий городов, добавленное примечание
df['city'] = df['city'].str.replace('не призн.', '', regex=False).str.strip()

In [13]:
# Проверю
df.query('region == "Крым"')

,city,region,population,found,status
22,Алупка,Крым,9063,960,1938
23,Алушта,Крым,31 364,VI век,1902
42,Армянск,Крым,20 692,1736,1993
76,Бахчисарай,Крым,28 609,1532,1532
86,Белогорск,Крым,17 445,XIII век,XIII век
250,Джанкой,Крым,37 014,1784,1917
274,Евпатория,Крым,107 877,497 год до н. э.,1784
397,Керчь,Крым,154 621,VII—VI век до н. э.,NaN
474,Красноперекопск,Крым,25 569,1932,1966
825,Саки,Крым,24 285,1952,1952


In [14]:
# Теперь проверю уникальные значения регионов
df['region'].unique()

array(['Хакасия', 'Оренбургская область', 'Краснодарский край',
       'Башкортостан', 'Татарстан', 'Адыгея', 'Ростовская область',
       'Тыва', 'Северная Осетия', 'Свердловская область', 'Чувашия',
       'Якутия', 'Алтайский край', 'Владимирская область',
       'Пермский край', 'Сахалинская область', 'Белгородская область',
       'Тульская область', 'Иркутская область', 'Крым',
       'Хабаровский край', 'Чукотский АО', 'Тверская область',
       'Кемеровская область', 'Мурманская область', 'Московская область',
       'Чечня', 'Мордовия', 'Нижегородская область',
       'Саратовская область', 'Приморский край', 'Красноярский край',
       'Архангельская область', 'Томская область', 'Астраханская область',
       'Челябинская область', 'Вологодская область', 'Бурятия',
       'Калининградская область', 'Кабардино-Балкария',
       'Калужская область', 'Забайкальский край', 'Новосибирская область',
       'Ульяновская область', 'Кировская область', 'Пензенская область',
       'Ам

АО это как правило автономный округ. При поиске координат, чтобы не было проблем, лучше исправлю.

In [15]:
# Заменю "АО"
df['region'] = df['region'].str.replace('АО', 'автономный округ', regex=False).str.strip()

In [16]:
# Проверю еще раз уникальные значения
df['region'].unique()

array(['Хакасия', 'Оренбургская область', 'Краснодарский край',
       'Башкортостан', 'Татарстан', 'Адыгея', 'Ростовская область',
       'Тыва', 'Северная Осетия', 'Свердловская область', 'Чувашия',
       'Якутия', 'Алтайский край', 'Владимирская область',
       'Пермский край', 'Сахалинская область', 'Белгородская область',
       'Тульская область', 'Иркутская область', 'Крым',
       'Хабаровский край', 'Чукотский автономный округ',
       'Тверская область', 'Кемеровская область', 'Мурманская область',
       'Московская область', 'Чечня', 'Мордовия', 'Нижегородская область',
       'Саратовская область', 'Приморский край', 'Красноярский край',
       'Архангельская область', 'Томская область', 'Астраханская область',
       'Челябинская область', 'Вологодская область', 'Бурятия',
       'Калининградская область', 'Кабардино-Балкария',
       'Калужская область', 'Забайкальский край', 'Новосибирская область',
       'Ульяновская область', 'Кировская область', 'Пензенская област

Как всегда ести исключения. "Еврейская автономный округ" нет такого региона. Исправлю.

In [17]:
df['region'] = df['region'].str.replace('Еврейская автономный округ', 'Еврейская автономная область', regex=False).str.strip()

In [18]:
# Проверю
df.query('region == "Еврейская автономная область"')

,city,region,population,found,status
104,Биробиджан,Еврейская автономная область,70 064,1915,1931
699,Облучье,Еврейская автономная область,7959,1911,1915


In [19]:
# Проверю дубликаты по городам
df['city'].value_counts().sort_values(ascending=False).head(20)

city
Советск           3
Михайловск        2
Белогорск         2
Заречный          2
Железногорск      2
Красноармейск     2
Радужный          2
Озёрск            2
Кировск           2
Киров             2
Благовещенск      2
Никольск          2
Мирный            2
Фокино            2
Краснознаменск    2
Гурьевск          2
Берёзовский       2
Краснослободск    2
Приморск          2
Алейск            1
Name: count, dtype: int64

19 городов с одинаковыми названиями. Добавляю столбец, объединяющий название города и регион, чтобы геоданные не задублировались. После получения геоданных столбец name можно будет удалить.

In [20]:
# добавляю столбец
df['name'] = df['city'] + ' ' + df['region']
df.head()

,city,region,population,found,status,name
0,Абаза,Хакасия,12 272,1867,1966,Абаза Хакасия
1,Абакан,Хакасия,184 769,1734,1931,Абакан Хакасия
2,Абдулино,Оренбургская область,17 274,1795,1923,Абдулино Оренбургская область
3,Абинск,Краснодарский край,39 511,1863,1963,Абинск Краснодарский край
4,Агидель,Башкортостан,14 219,1980,1991,Агидель Башкортостан


In [21]:
# Забираем координаты
# Инициируем геолокатор
# RateLimiter (ограничение скорости запросов) — паттерн, который позволяет предохранить сервис от перегрузки большим числом одновременных запросов.
# RateLimiter контролирует пропускную способность и при необходимости отбрасывает лишние запросы.
# Это предотвращает ситуацию, когда слишком много клиентов одновременно обращаются к одному сервису, делая его недоступным для всех
# User_Agent — это заголовок HTTP-запроса, который отправляется с каждым запросом.
# Nominatim требует, чтобы это значение было установлено как имя вашего приложения.
# Цель состоит в том, чтобы иметь возможность ограничивать количество запросов на одно приложение.
geolocator = Nominatim(user_agent="russia_locator")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # Ограничение 1 запрос в секунду

In [22]:
# Добавляем в датасет колонки с широтой и долготой
df['latitude'] = None
df['longitude'] = None
df.head()

,city,region,population,found,status,name,latitude,longitude
0,Абаза,Хакасия,12 272,1867,1966,Абаза Хакасия,None,None
1,Абакан,Хакасия,184 769,1734,1931,Абакан Хакасия,None,None
2,Абдулино,Оренбургская область,17 274,1795,1923,Абдулино Оренбургская область,None,None
3,Абинск,Краснодарский край,39 511,1863,1963,Абинск Краснодарский край,None,None
4,Агидель,Башкортостан,14 219,1980,1991,Агидель Башкортостан,None,None


In [24]:
# tqdm добавляет прогресс бар в итерацию
for idx, row in tqdm(df.iterrows(), total=len(df), desc='Загружаем координаты городов'):
    # Если координаты для города есть - пропускаем строку
    if pd.notna(row['latitude']) and pd.notna(row['longitude']):
        continue
    try:
        # Если координат нет - добавляем
        query = f"{row['name']}, {row['region']}, 'Россия'"
        location = geocode(query)
        if location:
            # Обновляем строку в DataFrame
            df.at[idx, 'latitude'] = location.latitude
            df.at[idx, 'longitude'] = location.longitude

        # Пауза между запросами
        time.sleep(1)

    # Обрабатываем все исключения
    except Exception as e:
        print(f"Ошибка для {row['city_name']}: {str(e)}")
        continue

Загружаем координаты городов: 100%|████████████████████████████████████████████████| 1125/1125 [21:28<00:00,  1.15s/it]


In [25]:
# Проверю
df.head()

,city,region,population,found,status,name,latitude,longitude
0,Абаза,Хакасия,12 272,1867,1966,Абаза Хакасия,52.651055,90.101159
1,Абакан,Хакасия,184 769,1734,1931,Абакан Хакасия,53.72068,91.440602
2,Абдулино,Оренбургская область,17 274,1795,1923,Абдулино Оренбургская область,53.69099,53.647504
3,Абинск,Краснодарский край,39 511,1863,1963,Абинск Краснодарский край,44.864953,38.157819
4,Агидель,Башкортостан,14 219,1980,1991,Агидель Башкортостан,55.898963,53.934191


In [26]:
# Проверим основные данные о файле с помощью функции
def date_info(date_table):
    '''Находим общую инфрмацию об исследуемом датафрейме с помощью функций info, describe, shape, duplicated()
    и выводим количество пропусков по столбцам с помощью функции for.
    '''

    print('\033[1m' + 'Общая информация о датафрейме и типы данных:' + '\033[0m')
    display(date_table.info())
    print('\033[1m' + 'Описательная статистика столбцов датафрейма методом describe:' + '\033[0m')
    display(date_table.describe())
    duplicate_dict = {}
    isnull_dict = {}
    for value in date_table.columns:
        duplicate_dict[value] = date_table[value].duplicated().sum()
        isnull_dict[value] = date_table[value].isnull().sum()
        tmp_df = pd.DataFrame([isnull_dict])
    tmp_df.index = ['Пропусков в столбце']
    tmp_df = tmp_df.style.map(lambda x: 'color:darkred' if x > 0 else 'color:dark')
    print('\033[1m' + 'Количество пропусков по столбцам:' + '\033[0m')
    display(tmp_df)
    '''опять'''
    print('\033[1m' + 'Количество строк = ' + '\033[0m', f'{date_table.shape[0]}')
    print('\033[1m' + 'Количество столбцов =' + '\033[0m', f'{date_table.shape[1]}\n')
    result_total = print('\033[1m' + 'Явных дубликатов в датафрейме =' + '\033[0m', date_table.duplicated().sum())
    return result_total
date_info(df)

Общая информация о датафрейме и типы данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1125 entries, 0 to 1124
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   city        1125 non-null   object
 1   region      1125 non-null   object
 2   population  1125 non-null   object
 3   found       1125 non-null   object
 4   status      1122 non-null   object
 5   name        1125 non-null   object
 6   latitude    1119 non-null   object
 7   longitude   1119 non-null   object
dtypes: object(8)
memory usage: 70.4+ KB


None

Описательная статистика столбцов датафрейма методом describe:


,city,region,population,found,status,name,latitude,longitude
count,1125,1125,1125,1125,1122,1125,1119.000000,1119.000000
unique,1105,85,1116,491,252,1125,1118.000000,1118.000000
top,Советск,Московская область,10 994,XVIII век,1938,Абаза Хакасия,49.765733,43.651608
freq,3,74,2,15,50,1,2.000000,2.000000


Количество пропусков по столбцам:


,city,region,population,found,status,name,latitude,longitude
Пропусков в столбце,0,0,0,0,3,0,6,6


Количество строк =  1125
Количество столбцов = 8

Явных дубликатов в датафрейме = 0


In [27]:
# Проверю координаты по Советску
df.query('city == "Советск"')

,city,region,population,found,status,name,latitude,longitude
885,Советск,Калининградская область,38 910,1288,1552,Советск Калининградская область,55.080746,21.888161
886,Советск,Кировская область,14 626,XII век,1937,Советск Кировская область,57.586189,48.958698
887,Советск,Тульская область,7889,1949,1954,Советск Тульская область,53.934659,37.632086


Координаты с одним названияем города разные.

In [28]:
# Теперь посмотрю 3 пропуска
df.loc[df['status'].isna() == True]

,city,region,population,found,status,name,latitude,longitude
397,Керчь,Крым,154 621,VII—VI век до н. э.,NaN,Керчь Крым,45.360651,36.434392
927,Судак,Крым,17 834,212,NaN,Судак Крым,None,None
1024,Феодосия,Крым,66 293,VI век до н. э.,NaN,Феодосия Крым,45.026237,35.386749


Судак [статус города](https://tavrida.crimealib.ru/view_selo.php?id=11#:~:text=%D0%92%201979%20%D0%B3.%20%D0%A1%D1%83%D0%B4%D0%B0%D0%BA%20%D0%BF%D0%BE%D0%BB%D1%83%D1%87%D0%B8%D0%BB%20%D1%81%D1%82%D0%B0%D1%82%D1%83%D1%81%20%D0%B3%D0%BE%D1%80%D0%BE%D0%B4%D0%B0) получил в 1979 году.

Феодосия [статус города](https://ru.wikipedia.org/wiki/%D0%A4%D0%B5%D0%BE%D0%B4%D0%BE%D1%81%D0%B8%D1%8F#:~:text=%D1%81%201787%C2%A0%D0%B3%D0%BE%D0%B4%D0%B0%20%D0%BF%D0%BE%D1%81%D0%B5%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5%20%D0%BF%D0%BE%D0%BB%D1%83%D1%87%D0%B8%D0%BB%D0%BE%20%D1%81%D1%82%D0%B0%D1%82%D1%83%D1%81%20%D0%B3%D0%BE%D1%80%D0%BE%D0%B4%D0%B0%5B8%5D.) получила в 1787 году.

Керчь [статус города](https://bigenc.ru/c/kerch-a311b0#:~:text=%D0%94%D0%B0%D1%82%D0%B0%20%D0%BF%D1%80%D0%B8%D1%81%D0%B2%D0%BE%D0%B5%D0%BD%D0%B8%D1%8F%20%D1%81%D1%82%D0%B0%D1%82%D1%83%D1%81%D0%B0,%D0%B4%D0%BE%20%D0%BD%D0%B0%D1%88%D0%B5%D0%B9%20%D1%8D%D1%80%D1%8B) получила в 7 веке до нашей эры.

In [29]:
# Добавим эти данные в таблицу
# Керчь
df.at[397, 'status'] = 'VII век до н. э.'
# Судак
df.at[927, 'status'] = '1979'
# Феодосия
df.at[1024, 'status'] = '1787'

In [30]:
# Проверю
display(df.query('city == "Керчь"'))
display(df.query('city == "Судак"'))
display(df.query('city == "Феодосия"'))

,city,region,population,found,status,name,latitude,longitude
397,Керчь,Крым,154 621,VII—VI век до н. э.,VII век до н. э.,Керчь Крым,45.360651,36.434392


,city,region,population,found,status,name,latitude,longitude
927,Судак,Крым,17 834,212,1979,Судак Крым,None,None


,city,region,population,found,status,name,latitude,longitude
1024,Феодосия,Крым,66 293,VI век до н. э.,1787,Феодосия Крым,45.026237,35.386749


In [31]:
# Удалю колонку name, т.к. она нам не нужна в дальнейшем.
df = df.drop('name', axis=1)
df.head()

,city,region,population,found,status,latitude,longitude
0,Абаза,Хакасия,12 272,1867,1966,52.651055,90.101159
1,Абакан,Хакасия,184 769,1734,1931,53.72068,91.440602
2,Абдулино,Оренбургская область,17 274,1795,1923,53.69099,53.647504
3,Абинск,Краснодарский край,39 511,1863,1963,44.864953,38.157819
4,Агидель,Башкортостан,14 219,1980,1991,55.898963,53.934191


In [32]:
# Контрольная проверка
print(date_info.__doc__)
date_info(df)

Находим общую инфрмацию об исследуемом датафрейме с помощью функций info, describe, shape, duplicated()
    и выводим количество пропусков по столбцам с помощью функции for.
    
Общая информация о датафрейме и типы данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1125 entries, 0 to 1124
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   city        1125 non-null   object
 1   region      1125 non-null   object
 2   population  1125 non-null   object
 3   found       1125 non-null   object
 4   status      1125 non-null   object
 5   latitude    1119 non-null   object
 6   longitude   1119 non-null   object
dtypes: object(7)
memory usage: 61.7+ KB


None

Описательная статистика столбцов датафрейма методом describe:


,city,region,population,found,status,latitude,longitude
count,1125,1125,1125,1125,1125,1119.000000,1119.000000
unique,1105,85,1116,491,254,1118.000000,1118.000000
top,Советск,Московская область,10 994,XVIII век,1938,49.765733,43.651608
freq,3,74,2,15,50,2.000000,2.000000


Количество пропусков по столбцам:


,city,region,population,found,status,latitude,longitude
Пропусков в столбце,0,0,0,0,0,6,6


Количество строк =  1125
Количество столбцов = 7

Явных дубликатов в датафрейме = 0


Можно сохранять.

In [33]:
df.to_csv('Goroda_Russia_all.csv', encoding='utf8', index=False)